In [ ]:
import cv2                     # OpenCV for image processing (HE, CLAHE)
import numpy as np            # Numerical operations
import tensorflow as tf       # Deep learning framework
from tensorflow.keras import layers, models, utils
import matplotlib.pyplot as plt  # Plotting graphs


# =========================
# KAGGLE SETUP
# =========================

# Install Kaggle API (used to download dataset)
!pip install -q kaggle

# Upload kaggle.json (API key file)
from google.colab import files
files.upload()

# Create kaggle directory
!mkdir -p ~/.kaggle

# Move API key to correct location
!mv kaggle.json ~/.kaggle/

# Set proper permissions (required by Kaggle)
!chmod 600 ~/.kaggle/kaggle.json

# Download Intel Image Classification dataset
!kaggle datasets download -d puneet6060/intel-image-classification

# Unzip dataset
!unzip -q intel-image-classification.zip


# =========================
# PREPROCESSING FUNCTIONS
# =========================

# Histogram Equalization (global contrast enhancement)
def tf_he(images, labels):
    images = tf.cast(images, tf.uint8)  # Convert images to uint8

    def he(img):
        # Convert RGB → YCrCb (separates luminance from color)
        ycrcb = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)

        # Apply histogram equalization on luminance channel
        ycrcb[:,:,0] = cv2.equalizeHist(ycrcb[:,:,0])

        # Convert back to RGB
        return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)

    # Apply HE to each image in batch
    images = tf.map_fn(
        lambda x: tf.numpy_function(he, [x], tf.uint8),
        images
    )

    # Fix shape (lost after numpy_function)
    images.set_shape([None, 128, 128, 3])

    return images, labels


# CLAHE (adaptive histogram equalization)
def tf_clahe(images, labels):
    images = tf.cast(images, tf.uint8)

    def clahe(img):
        ycrcb = cv2.cvtColor(img, cv2.COLOR_RGB2YCrCb)

        # Apply CLAHE on luminance channel
        ycrcb[:,:,0] = cv2.createCLAHE(2.0, (8,8)).apply(ycrcb[:,:,0])

        return cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)

    # Apply CLAHE to batch
    images = tf.map_fn(
        lambda x: tf.numpy_function(clahe, [x], tf.uint8),
        images
    )

    # Fix shape
    images.set_shape([None, 128, 128, 3])

    return images, labels


# =========================
# DATA LOADING
# =========================

IMG_SIZE = (128, 128)   # Resize all images to 128x128
BATCH_SIZE = 32         # Number of images per batch

# Load training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/seg_train/seg_train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Load validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    "/content/seg_test/seg_test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False  # Keep order fixed for evaluation
)

# Create different dataset variants
train_raw = train_ds              # No preprocessing
train_he = train_ds.map(tf_he)   # Apply Histogram Equalization
train_clahe = train_ds.map(tf_clahe)  # Apply CLAHE

val_raw = val_ds
val_he = val_ds.map(tf_he)
val_clahe = val_ds.map(tf_clahe)


# =========================
# CLASS NAMES
# =========================

# Extract class labels from dataset
class_names = train_ds.class_names
print(class_names)


# =========================
# RESNET BLOCK
# =========================

def residual_block(x, filters, downsample=False):

    shortcut = x  # Save input for skip connection
    stride = 2 if downsample else 1  # Downsample if needed

    # First convolution
    x = layers.Conv2D(filters, 3, strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Second convolution
    x = layers.Conv2D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)

    # Adjust shortcut if dimensions mismatch
    if downsample or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    # Add skip connection
    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)

    return x


# =========================
# MODEL DEFINITION
# =========================

def build_resnet(input_shape=(128,128,3), num_classes=6):

    inputs = layers.Input(shape=input_shape)

    # Normalize pixel values (0–255 → 0–1)
    x = layers.Rescaling(1./255)(inputs)

    # Initial convolution
    x = layers.Conv2D(32, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Residual blocks
    x = residual_block(x, 32)
    x = residual_block(x, 64, downsample=True)
    x = residual_block(x, 64)
    x = residual_block(x, 128, downsample=True)
    x = residual_block(x, 128)

    # Global feature aggregation
    x = layers.GlobalAveragePooling2D()(x)

    # Fully connected layer
    x = layers.Dense(128, activation='relu')(x)

    # Dropout for regularization
    x = layers.Dropout(0.3)(x)

    # Output layer (6 classes)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)


# ============================================================
# EXPERIMENT 1: CLAHE → CLAHE
# ============================================================

model = build_resnet()

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train model
history = model.fit(
    train_clahe,
    validation_data=val_clahe,
    epochs=12
)

# Plot accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('CLAHE → CLAHE Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'validation'])
plt.show()

# Plot loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('CLAHE → CLAHE Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'validation'])
plt.show()

# Evaluation
y_true = []
y_pred = []

# Predict on validation set
for x, y in val_clahe:
    preds = model.predict(x, verbose=0)
    y_pred.extend(preds.argmax(axis=1))
    y_true.extend(y.numpy())

# Metrics
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


# ============================================================
# EXPERIMENT 2: RAW → RAW
# ============================================================

model = build_resnet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history1 = model.fit(
    train_raw,
    validation_data=val_raw,
    epochs=12
)

plt.plot(history1.history['accuracy'])
plt.plot(history1.history['val_accuracy'])
plt.title('RAW → RAW Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'validation'])
plt.show()

plt.plot(history1.history['loss'])
plt.plot(history1.history['val_loss'])
plt.title('RAW → RAW Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'validation'])
plt.show()

y_true = []
y_pred = []

for x, y in val_raw:
    preds = model.predict(x, verbose=0)
    y_pred.extend(preds.argmax(axis=1))
    y_true.extend(y.numpy())

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


# ============================================================
# EXPERIMENT 3: HE → HE
# ============================================================

model = build_resnet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_he,
    validation_data=val_he,
    epochs=12
)

plt.plot(history2.history['accuracy'])
plt.plot(history2.history['val_accuracy'])
plt.title('HE → HE Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'validation'])
plt.show()

plt.plot(history2.history['loss'])
plt.plot(history2.history['val_loss'])
plt.title('HE → HE Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'validation'])
plt.show()

y_true = []
y_pred = []

for x, y in val_he:
    preds = model.predict(x, verbose=0)
    y_pred.extend(preds.argmax(axis=1))
    y_true.extend(y.numpy())

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


# ============================================================
# EXPERIMENT 4: CLAHE → RAW
# ============================================================

model = build_resnet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history3 = model.fit(
    train_clahe,
    validation_data=val_raw,
    epochs=12
)

plt.plot(history3.history['accuracy'])
plt.plot(history3.history['val_accuracy'])
plt.title('CLAHE → RAW Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'validation'])
plt.show()

plt.plot(history3.history['loss'])
plt.plot(history3.history['val_loss'])
plt.title('CLAHE → RAW Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'validation'])
plt.show()

y_true = []
y_pred = []

for x, y in val_raw:
    preds = model.predict(x, verbose=0)
    y_pred.extend(preds.argmax(axis=1))
    y_true.extend(y.numpy())

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))


# ============================================================
# EXPERIMENT 5: HE → RAW
# ============================================================

model = build_resnet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history4 = model.fit(
    train_he,
    validation_data=val_raw,
    epochs=12
)

plt.plot(history4.history['accuracy'])
plt.plot(history4.history['val_accuracy'])
plt.title('HE → RAW Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(['train', 'validation'])
plt.show()

plt.plot(history4.history['loss'])
plt.plot(history4.history['val_loss'])
plt.title('HE → RAW Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(['train', 'validation'])
plt.show()

y_true = []
y_pred = []

for x, y in val_raw:
    preds = model.predict(x, verbose=0)
    y_pred.extend(preds.argmax(axis=1))
    y_true.extend(y.numpy())

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=class_names))